In [ ]:
# clone repo and install package
!git clone https://github.com/rpw1134/music-generation.git
!pip install -e music-generation --quiet

In [1]:
import torch
from torch.utils.data import DataLoader, random_split

from midi_gen.model.models.GPTMidiV1 import GPTMidiV1
from midi_gen.model.training.data import TokenDataset
from midi_gen.model.training.training_loop import training_loop, load_checkpoint

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.11.0
CUDA available: False


In [2]:
# paths — dataset should be uploaded as a Kaggle Dataset and attached to this notebook
DATASET_PATH = "/kaggle/input/midi-tokenized/tokenized_dataset.npy"
CHECKPOINT_PATH = "/kaggle/working/midiv1_best.pt"

In [4]:
# hyperparameters
BATCH_SIZE = 32
NUM_EPOCHS = 10
LR = 3e-4
WARMUP_STEPS = 200
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
VAL_SPLIT = 0.1

# model
D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 6
FF_DIM_RATIO = 4
DROPOUT = 0.1

In [5]:
# dataset and train/val split
dataset = TokenDataset(DATASET_PATH)
val_size = int(len(dataset) * VAL_SPLIT)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(f"Train sequences: {train_size}")
print(f"Val sequences:   {val_size}")
print(f"Steps per epoch: {len(train_loader)}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/midi-tokenized/tokenized_dataset.npy'

In [ ]:
# model
model = GPTMidiV1(
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ff_dim_ratio=FF_DIM_RATIO,
    dropout=DROPOUT
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")

In [ ]:
# train
history = training_loop(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    grad_clip=GRAD_CLIP,
    checkpoint_path=CHECKPOINT_PATH
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history["accuracy"])
axes[1].set_title("Val Accuracy")

axes[2].plot(history["perplexity"])
axes[2].set_title("Perplexity")

plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png")
plt.show()
print("Saved training_curves.png")

In [ ]:
# confirm output files are ready for download
import os
for f in os.listdir("/kaggle/working/"):
    size_mb = os.path.getsize(f"/kaggle/working/{f}") / 1e6
    print(f"{f}  ({size_mb:.1f} MB)")